# EfficientNetV2-S on HAM10000 — Colab TPU training

**How to run:**
1. Open this notebook in Google Colab (upload it, or open via the Colab extension).
2. **Runtime → Change runtime type → TPU**, then connect.
3. Run all cells top to bottom. You will be asked to authorize Google Drive access (checkpoints and logs are saved there so nothing is lost if the runtime disconnects).
4. First run streams the HAM10000 subset from Hugging Face (~10 min) and caches it to Drive; later runs restore from the cache in seconds.

The default backbone is **EfficientNetV2-S @ 384px** (change `MODEL_NAME` in the config cell to try ConvNeXt, Swin, ViT, …). Progress is printed every 20 batches and every epoch summary is also appended to `MyDrive/galaxy-uq/results/skin/progress.log`, so you can check progress from any device even if this tab disconnects.

# EfficientNet Training on HAM10000 with Colab TPU

Fine-tune a torchvision EfficientNet on the HAM10000 skin lesion dataset using
PyTorch/XLA on a free-tier Colab TPU. The default is **EfficientNetV2-S @ 384px**
on the full training set, fine-tuned for up to 40 epochs with **early stopping**.
Falls back automatically to GPU/CPU if no TPU is attached.

**Just run all cells once** (Runtime → TPU → Run all). Everything is configured;
the run downloads the data (one-time, cached to Drive), trains, evaluates, and
saves the model + plots to Drive.

### Knobs to experiment with (all in the config cell)

Transfer learning happens in two phases — train the new head briefly (backbone
frozen), then fine-tune the whole network at a low learning rate — plus an optional
third phase that rebalances the classifier (see *Combating class imbalance* in the
README). Things worth tweaking to build intuition:

- `MODEL_NAME` — swap the backbone (`convnext_small`, `swin_v2_t`, `vit_b_16`, …).
  The head-swap is generic, so any torchvision classifier works.
- `FT_LR` / `HEAD_LR` — fine-tuning learning rates. Too high wipes the pretrained
  features; too low barely moves them.
- `FT_EPOCHS` / `EARLY_STOP_PATIENCE` — how long to train and how patient to be
  before stopping when validation stops improving.
- `LOGIT_ADJUST_TAU` — strength of the logit-adjusted loss (the imbalance corrector
  for phases 1–2). `1.0` is standard; `0` falls back to class-weighted CE.
- `CRT_EPOCHS` / `CRT_LR` — epochs/LR of phase-3 decoupled classifier re-training on
  a class-balanced sampler. `0` skips it.
- `FULL_DATASET` — all images vs. a per-class cap that rebalances the classes.

## 1. Setup TPU and Environment

In [ ]:
# Run this first on a fresh runtime. Installs PyTorch/XLA (TPU support) and the
# Hugging Face datasets library used to download HAM10000.
import os

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

IS_TPU_RUNTIME = IN_COLAB and bool(
    os.environ.get("COLAB_TPU_1VM") or os.environ.get("TPU_ACCELERATOR_TYPE")
)
print(f"Colab: {IN_COLAB}  TPU runtime: {IS_TPU_RUNTIME}")

if IN_COLAB:
    !pip install -q datasets

if IS_TPU_RUNTIME:
    import torch
    TORCH_VER = torch.__version__.split("+")[0]
    # torch_xla must match the preinstalled torch version exactly
    !pip install -q "torch_xla[tpu]~={TORCH_VER}" -f https://storage.googleapis.com/libtpu-releases/index.html -f https://storage.googleapis.com/libtpu-wheels/index.html
elif IN_COLAB:
    print("WARNING: this is not a TPU runtime — training will fall back to GPU/CPU.")
    print("Use Runtime -> Change runtime type -> TPU, then rerun from the top.")

In [ ]:
import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import balanced_accuracy_score, confusion_matrix
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, models, transforms

print(f"PyTorch version: {torch.__version__}")

In [ ]:
# Device setup: TPU via PyTorch/XLA, otherwise CUDA/MPS/CPU
USE_TPU = False
if IS_TPU_RUNTIME:
    import torch_xla
    import torch_xla.core.xla_model as xm
    import torch_xla.distributed.parallel_loader as pl
    # torch_xla.device() replaces the deprecated xm.xla_device()
    device = torch_xla.device() if hasattr(torch_xla, "device") else xm.xla_device()
    USE_TPU = True
else:
    device = torch.device(
        "cuda" if torch.cuda.is_available()
        else "mps" if torch.backends.mps.is_available()
        else "cpu"
    )
print(f"Device: {device}  (TPU: {USE_TPU})")


def device_loader(loader):
    """On TPU, MpDeviceLoader prefetches batches to the device and inserts the
    XLA mark_step after every batch — without it the lazy graph never executes."""
    return pl.MpDeviceLoader(loader, device) if USE_TPU else loader


def save_checkpoint(state_dict, path):
    if USE_TPU:
        xm.save(state_dict, str(path))  # moves XLA tensors to CPU before writing
    else:
        torch.save(state_dict, path)

In [ ]:
# Dataset variant: per-class caps + stored resolution. These are baked into the
# cache name so changing them can never accidentally reuse a stale Drive cache.
FULL_DATASET = True   # True = use every training image (recommended for EfficientNetV2-S).
                      # The majority class (melanocytic_nevi, ~60% of HAM10000) then
                      # dominates, but the class-weighted loss + balanced-accuracy metric
                      # compensate. Set False to cap each class and rebalance the train set.
TRAIN_CAP = 100_000 if FULL_DATASET else 2000  # 100k = effectively uncapped
VAL_CAP = 200         # keep validation capped/balanced for a clean, comparable metric
SHORT_SIDE = 448      # stored image short side; must exceed the val resize (~439 at
                      # IMG_SIZE=384) so EfficientNetV2-S inputs are never upscaled

VARIANT = f"skin_t{TRAIN_CAP}_v{VAL_CAP}_s{SHORT_SIDE}"

# Paths: outputs go to Drive (survive disconnects), image data stays on the
# local VM disk (reading thousands of small files from Drive is very slow).
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = Path('/content/drive/MyDrive/galaxy-uq')
    OUT_DIR = DRIVE_DIR / 'results' / 'skin'
    DATA_DIR = Path('/content/data') / VARIANT
    DATA_CACHE_TAR = DRIVE_DIR / 'data' / f'{VARIANT}.tar.gz'
else:
    SKIN = Path.cwd()  # run from the skin/ directory
    DATA_DIR = SKIN / 'data'
    OUT_DIR = SKIN / 'results'
    DATA_CACHE_TAR = None

OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Data dir:   {DATA_DIR}")
print(f"Output dir: {OUT_DIR}")

## 2. Download HAM10000 Dataset

Streams the `marmal88/skin_cancer` HAM10000 mirror from Hugging Face (same logic as `skin/01_data.py`: capped per class to rebalance, short side resized to 256px). The first run takes ~10 min and caches a tarball to Drive; later runs restore from that cache in seconds.

In [ ]:
if (DATA_DIR / "train").exists():
    print(f"Dataset already present at {DATA_DIR}")
elif DATA_CACHE_TAR is not None and DATA_CACHE_TAR.exists():
    print(f"Restoring dataset from Drive cache: {DATA_CACHE_TAR}")
    DATA_DIR.parent.mkdir(parents=True, exist_ok=True)
    !tar -xzf "{DATA_CACHE_TAR}" -C "{DATA_DIR.parent}"
else:
    from datasets import load_dataset
    from PIL import Image
    from tqdm.auto import tqdm

    def resize_short_side(img, target=SHORT_SIDE):
        w, h = img.size
        scale = target / min(w, h)
        if scale >= 1.0:
            return img
        return img.resize((round(w * scale), round(h * scale)), Image.LANCZOS)

    def harvest(split, out_name, cap):
        ds = load_dataset("marmal88/skin_cancer", split=split, streaming=True)
        counts = {}
        for ex in tqdm(ds, desc=f"{split} -> {out_name}", unit="img"):
            dx = ex["dx"].lower()
            if counts.get(dx, 0) >= cap:
                continue
            counts[dx] = counts.get(dx, 0) + 1
            out_dir = DATA_DIR / out_name / dx
            out_dir.mkdir(parents=True, exist_ok=True)
            img = resize_short_side(ex["image"].convert("RGB"))
            img.save(out_dir / f"{ex['image_id']}.jpg", quality=87)
        print(f"{out_name} per-class counts: {counts}")

    print("Streaming HAM10000 subset from Hugging Face (one-time, ~15 min)...")
    harvest("train", "train", TRAIN_CAP)
    harvest("validation", "val", VAL_CAP)

    if DATA_CACHE_TAR is not None:
        DATA_CACHE_TAR.parent.mkdir(parents=True, exist_ok=True)
        !tar -czf "{DATA_CACHE_TAR}" -C "{DATA_DIR.parent}" "{DATA_DIR.name}"
        print(f"Cached dataset to {DATA_CACHE_TAR}")

for split in ["train", "val"]:
    split_dir = DATA_DIR / split
    n = sum(1 for _ in split_dir.rglob("*.jpg")) if split_dir.exists() else 0
    print(f"  {split}: {n} images")

## 3. Training Configuration

In [ ]:
# Training hyperparameters
MODEL_NAME = "efficientnet_v2_s"  # any torchvision classifier: convnext_*, swin_v2_*, vit_b_16, efficientnet_b3, ...
IMG_SIZE = 384                    # EfficientNetV2-S native resolution (keep SHORT_SIDE >= the val resize)
VAL_RESIZE = round(IMG_SIZE * 256 / 224)  # keep the same resize/crop ratio as the 224 recipe
BATCH_SIZE = 24                   # tuned for V2-S @ 384 on a TPU core; lower to 16 if OOM, raise to 32 if memory allows
NUM_WORKERS = min(8, os.cpu_count() or 2)  # Colab TPU VMs have plenty of CPU cores

# Two-phase transfer learning. Phase 1 trains only the new head briefly; phase 2
# fine-tunes the whole network. Phase 2 runs up to FT_EPOCHS but stops early once
# validation balanced accuracy hasn't improved for EARLY_STOP_PATIENCE epochs, so
# setting FT_EPOCHS high is safe — you pay only for the epochs that actually help.
HEAD_EPOCHS = 3
FT_EPOCHS = 40
EARLY_STOP_PATIENCE = 6           # stop phase 2 after this many epochs with no val improvement
HEAD_LR = 1e-3
FT_LR = 1e-4

# --- class-imbalance handling (see README "Combating class imbalance") ------
# HAM10000 is ~67% melanocytic nevi, so a plain cross-entropy run is biased
# toward the majority class. Two modern, complementary correctors are wired in:
#
#   1. Logit adjustment (Menon et al., ICLR 2021). Adds tau * log(class prior)
#      to the logits *inside the loss*. At inference the raw logits are then
#      Bayes-corrected for the label distribution, with no app/analysis change.
#      It replaces inverse-frequency class weighting (combining the two
#      double-corrects), so when tau > 0 the weighted CE is switched off.
#      tau = 1.0 is the standard, theoretically-motivated setting; 0 disables it.
#
#   2. Decoupled classifier re-training / cRT (Kang et al., ICLR 2020). After the
#      natural-distribution fine-tune, freeze the backbone and re-train ONLY the
#      head for a few epochs with a class-balanced sampler, rebalancing the
#      decision boundary without disturbing the learned features. CRT_EPOCHS = 0
#      disables this phase. The cRT phase uses plain (unweighted) CE because the
#      balanced sampler already equalises the classes.
#
# The two are independent; set one of them to 0 to ablate. Phase 3 only replaces
# the checkpoint if it beats phase 2's best val balanced accuracy, so enabling
# cRT can never make the saved model worse.
LOGIT_ADJUST_TAU = 1.0            # logit-adjusted loss strength (0 disables)
CRT_EPOCHS = 5                    # epochs of class-balanced classifier re-training (0 disables)
CRT_LR = 1e-3                     # LR for the cRT head (frozen backbone, so a head-sized LR)

# Compute saver: if a checkpoint for this MODEL_NAME already exists on Drive, the
# training cell loads it instead of retraining. Flip to True to force a fresh run.
FORCE_RETRAIN = False

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

print(f"Model: {MODEL_NAME} @ {IMG_SIZE}px  batch {BATCH_SIZE}  workers {NUM_WORKERS}")
print(f"Head: {HEAD_EPOCHS} epochs @ LR={HEAD_LR}")
print(f"Fine-tune: up to {FT_EPOCHS} epochs @ LR={FT_LR}, early stop patience {EARLY_STOP_PATIENCE}")
print(f"Imbalance: logit-adjust tau={LOGIT_ADJUST_TAU}  cRT epochs={CRT_EPOCHS} @ LR={CRT_LR}")

## 4. Data Loaders

In [ ]:
def make_loaders() -> tuple:
    train_tf = transforms.Compose([
        transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),  # lesions have no canonical orientation
        transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])
    val_tf = transforms.Compose([
        transforms.Resize(VAL_RESIZE),
        transforms.CenterCrop(IMG_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])

    train_ds = datasets.ImageFolder(DATA_DIR / "train", transform=train_tf)
    val_ds = datasets.ImageFolder(DATA_DIR / "val", transform=val_tf)

    assert train_ds.classes == val_ds.classes, "Train/val classes mismatch"

    # drop_last keeps batch shapes fixed — a ragged final batch forces an XLA
    # recompilation every epoch on TPU
    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
        persistent_workers=True, drop_last=True,
    )
    val_loader = DataLoader(
        val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
        persistent_workers=True,
    )

    return train_loader, val_loader, train_ds.classes

train_loader, val_loader, classes = make_loaders()
print(f"Classes: {classes}")
print(f"Train: {len(train_loader.dataset)}  Val: {len(val_loader.dataset)}")

## 5. Model Setup

In [ ]:
def class_weights(train_ds):
    counts = np.bincount([y for _, y in train_ds.samples])
    weights = counts.sum() / (len(counts) * counts)
    return torch.tensor(weights, dtype=torch.float32)


def log_class_prior(train_ds):
    """log P(y) over the training set — the offset logit adjustment subtracts."""
    counts = np.bincount([y for _, y in train_ds.samples])
    prior = counts / counts.sum()
    return torch.log(torch.tensor(prior, dtype=torch.float32).clamp_min(1e-12))


class LogitAdjustedLoss(nn.Module):
    """Logit-adjusted cross-entropy (Menon et al., ICLR 2021).

    Trains on ``logits + tau * log_prior``, which is equivalent to enforcing a
    per-class margin proportional to the label frequency. Because the prior is
    added during training, the *raw* logits at inference are already corrected
    for the class imbalance — so the app and analysis need no change. This is an
    alternative to inverse-frequency class weighting, not an addition to it."""
    def __init__(self, log_prior, tau):
        super().__init__()
        self.register_buffer("adj", tau * log_prior)

    def forward(self, logits, target):
        return F.cross_entropy(logits + self.adj, target)


def balanced_sampler(train_ds):
    """A WeightedRandomSampler that draws each class with equal probability, for
    the cRT phase. Sampling (rather than capping) keeps every image available
    while equalising how often each class is seen per epoch."""
    labels = [y for _, y in train_ds.samples]
    counts = np.bincount(labels)
    per_class_w = 1.0 / np.maximum(counts, 1)
    sample_w = [per_class_w[y] for y in labels]
    return WeightedRandomSampler(sample_w, num_samples=len(labels), replacement=True)


def replace_head(model, n_classes):
    """Swap the final classification layer for a fresh n_classes one.

    Different torchvision families name the head differently, so we locate the
    last nn.Linear generically. Works for EfficientNet (.classifier), ConvNeXt
    (.classifier), Swin (.head), and ViT (.heads). 'features' (frozen in phase 1)
    stays valid for EfficientNet/ConvNeXt; for Swin/ViT the phase-1 freeze below
    falls back to freezing everything except the new head."""
    # find the attribute path to the last Linear in the head container
    for head_attr in ["classifier", "head", "heads", "fc"]:
        head = getattr(model, head_attr, None)
        if head is None:
            continue
        if isinstance(head, nn.Linear):
            setattr(model, head_attr, nn.Linear(head.in_features, n_classes))
            return model
        # head is a Sequential / module: replace the last Linear inside it
        last_linear_name = None
        for name, m in head.named_modules():
            if isinstance(m, nn.Linear):
                last_linear_name = name
        if last_linear_name is not None:
            parent = head
            *path, leaf = last_linear_name.split(".")
            for p in path:
                parent = getattr(parent, p)
            in_features = getattr(parent, leaf).in_features
            setattr(parent, leaf, nn.Linear(in_features, n_classes))
            return model
    raise ValueError(f"Could not find a classifier head on {type(model).__name__}")


def freeze_backbone(model, freeze=True):
    """Phase-1 helper: freeze everything except the classification head."""
    head_params = set()
    for head_attr in ["classifier", "head", "heads", "fc"]:
        head = getattr(model, head_attr, None)
        if head is not None:
            head_params.update(id(p) for p in head.parameters())
    for p in model.parameters():
        p.requires_grad = (id(p) in head_params) if freeze else True


def head_parameters(model):
    for head_attr in ["classifier", "head", "heads", "fc"]:
        head = getattr(model, head_attr, None)
        if head is not None:
            return head.parameters()
    return model.parameters()


model = models.get_model(MODEL_NAME, weights="IMAGENET1K_V1")
model = replace_head(model, len(classes))
model.to(device)

# Phase 1/2 loss: logit adjustment if enabled, else inverse-frequency weighted CE.
# (Using both would double-correct the imbalance, so they are mutually exclusive.)
if LOGIT_ADJUST_TAU > 0:
    criterion = LogitAdjustedLoss(log_class_prior(train_loader.dataset), LOGIT_ADJUST_TAU).to(device)
    print(f"Loss: logit-adjusted CE (tau={LOGIT_ADJUST_TAU})")
else:
    criterion = nn.CrossEntropyLoss(weight=class_weights(train_loader.dataset).to(device))
    print("Loss: inverse-frequency class-weighted CE")
history = []

CKPT_PATH = OUT_DIR / f"{MODEL_NAME}_best.pt"

# the Gradio app (skin/03_app.py) reads this to rebuild the right architecture
with open(OUT_DIR / "model_config.json", "w") as f:
    json.dump({"model": MODEL_NAME, "img_size": IMG_SIZE, "checkpoint": CKPT_PATH.name}, f, indent=2)

print(f"Model: {MODEL_NAME}, output classes: {len(classes)}")
print(f"Checkpoint will be saved to: {CKPT_PATH}")

## 6. Evaluation Function

In [ ]:
def evaluate(model, loader):
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for x, y in device_loader(loader):
            logits = model(x.to(device))
            preds.append(logits.argmax(1).cpu().numpy())
            targets.append(y.cpu().numpy())
    preds, targets = np.concatenate(preds), np.concatenate(targets)
    return preds, targets, balanced_accuracy_score(targets, preds)

## 7. Training

Up to three phases: (1) head-only warm-up, (2) a full fine-tune at a low LR with early
stopping, and — when `CRT_EPOCHS > 0` — (3) **decoupled classifier re-training (cRT)**:
the backbone is frozen again and only the head is re-trained on a class-balanced sampler,
rebalancing the decision boundary for the imbalanced classes. The imbalance is also handled
inside phases 1–2 by the **logit-adjusted loss** (`LOGIT_ADJUST_TAU`). See the README's
*Combating class imbalance* section.

**Note on TPU:** the first few batches of each phase trigger XLA compilation, so expect the first progress line to take a couple of minutes — speed is normal after that.

Progress is printed every `LOG_EVERY` batches; epoch summaries also go to `progress.log` on Drive so you can check from anywhere.

In [ ]:
LOG_EVERY = 20  # batches between progress prints


def log_line(msg):
    """Print progress and append it to a log file on Drive, so training can be
    monitored even if the Colab tab disconnects."""
    stamp = time.strftime("%H:%M:%S")
    print(f"[{stamp}] {msg}", flush=True)
    with open(OUT_DIR / "progress.log", "a") as f:
        f.write(f"[{stamp}] {msg}\n")


def train_epochs(model, loader, val_loader, criterion, optimizer, epochs, tag, history,
                 patience=None, best=-1.0):
    """Train for up to `epochs`. If `patience` is set, stop early once validation
    balanced accuracy hasn't improved for that many consecutive epochs. The best
    checkpoint is saved to Drive whenever it improves, so an early stop (or a
    disconnect) always leaves the best weights on disk. `best` lets a later phase
    continue from an earlier phase's best score rather than resetting it."""
    since_improve = 0
    n_batches = len(loader)
    for epoch in range(epochs):
        model.train()
        # keep the running loss on device: calling .item() every batch would
        # stall the TPU pipeline, so we only sync at the log points
        running = torch.zeros((), device=device)
        n = 0
        t0 = time.time()
        for i, (x, y) in enumerate(device_loader(loader)):
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            optimizer.step()
            running += loss.detach() * len(y)
            n += len(y)
            if (i + 1) % LOG_EVERY == 0 or (i + 1) == n_batches:
                per_batch = (time.time() - t0) / (i + 1)
                eta = per_batch * (n_batches - i - 1)
                print(f"  [{tag} {epoch + 1}/{epochs}] batch {i + 1}/{n_batches}  "
                      f"loss {running.item() / n:.4f}  ~{eta:.0f}s left in epoch",
                      flush=True)
        _, _, bal_acc = evaluate(model, val_loader)
        epoch_loss = running.item() / n
        history.append({"phase": tag, "epoch": epoch, "loss": epoch_loss, "val_bal_acc": bal_acc})
        log_line(f"[{tag}] epoch {epoch + 1}/{epochs}  loss {epoch_loss:.4f}  "
                 f"val bal-acc {bal_acc:.3f}  ({time.time() - t0:.0f}s)")
        if bal_acc > best:
            best = bal_acc
            since_improve = 0
            save_checkpoint(model.state_dict(), CKPT_PATH)
            log_line(f"  -> new best checkpoint saved (val bal-acc {bal_acc:.3f})")
        else:
            since_improve += 1
            if patience is not None and since_improve >= patience:
                log_line(f"  -> early stop: no val improvement in {patience} epochs "
                         f"(best {best:.3f})")
                break
    return best


# Compute saver: skip the whole run if this model is already trained on Drive.
LOG_PATH = OUT_DIR / "training_log.json"
already_trained = CKPT_PATH.exists() and LOG_PATH.exists() and not FORCE_RETRAIN

if already_trained:
    prev = json.load(open(LOG_PATH))
    if prev.get("model") == MODEL_NAME:
        history = prev["history"]
        best = prev["best_val_bal_acc"]
        log_line(f"Found existing {MODEL_NAME} (val bal-acc {best:.3f}) — skipping training. "
                 f"Set FORCE_RETRAIN=True to retrain.")
    else:
        already_trained = False  # log belongs to a different model; train fresh

if not already_trained:
    log_line(f"Training started: {MODEL_NAME} @ {IMG_SIZE}px, "
             f"{len(train_loader.dataset)} train / {len(val_loader.dataset)} val images on {device}")

    # Phase 1: freeze backbone, train head only (generic across CNN/transformer heads)
    log_line("=== Phase 1: head training ===")
    freeze_backbone(model, freeze=True)
    opt = torch.optim.AdamW(head_parameters(model), lr=HEAD_LR)
    best = train_epochs(model, train_loader, val_loader, criterion, opt, HEAD_EPOCHS, "head", history)

    # Phase 2: unfreeze everything, fine-tune at low LR with early stopping
    log_line("=== Phase 2: full fine-tuning ===")
    freeze_backbone(model, freeze=False)
    opt = torch.optim.AdamW(model.parameters(), lr=FT_LR)
    best = train_epochs(model, train_loader, val_loader, criterion, opt, FT_EPOCHS, "ft", history,
                        patience=EARLY_STOP_PATIENCE, best=best)

    # Phase 3: decoupled classifier re-training (cRT). Restore phase-2's best
    # weights, freeze the backbone, and re-train only the head on a class-balanced
    # sampler with plain CE. This rebalances the decision boundary on top of the
    # features learned from the natural distribution. It carries `best` forward, so
    # the checkpoint is only overwritten if cRT actually improves val balanced
    # accuracy — enabling it can never regress the saved model.
    if CRT_EPOCHS > 0:
        log_line("=== Phase 3: classifier re-training (cRT, class-balanced) ===")
        model.load_state_dict(torch.load(CKPT_PATH, map_location="cpu", weights_only=True))
        model.to(device)
        # drop_last keeps batch shapes fixed for XLA; sampler replaces shuffle
        crt_loader = DataLoader(
            train_loader.dataset, batch_size=BATCH_SIZE,
            sampler=balanced_sampler(train_loader.dataset),
            num_workers=NUM_WORKERS, persistent_workers=True, drop_last=True,
        )
        crt_criterion = nn.CrossEntropyLoss()  # sampler balances classes, so no weights
        freeze_backbone(model, freeze=True)
        opt = torch.optim.AdamW(head_parameters(model), lr=CRT_LR)
        best = train_epochs(model, crt_loader, val_loader, crt_criterion, opt, CRT_EPOCHS,
                            "crt", history, best=best)

    log_line(f"Training done. Best val balanced accuracy: {best:.3f}")

## 8. Final Evaluation & Visualization

In [ ]:
# Load best checkpoint and evaluate (checkpoint is saved as CPU tensors)
model.load_state_dict(
    torch.load(CKPT_PATH, map_location="cpu", weights_only=True)
)
model.to(device)
preds, targets, bal_acc = evaluate(model, val_loader)

print(f"Final validation balanced accuracy: {bal_acc:.3f}")

In [ ]:
# Plot confusion matrix
cm = confusion_matrix(targets, preds, normalize="true")
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(classes)), classes, rotation=45, ha="right", fontsize=9)
ax.set_yticks(range(len(classes)), classes, fontsize=9)
for i in range(len(classes)):
    for j in range(len(classes)):
        ax.text(j, i, f"{cm[i, j]:.2f}", ha="center", va="center",
                color="white" if cm[i, j] > 0.5 else "black", fontsize=8)
ax.set_xlabel("Predicted", fontsize=10)
ax.set_ylabel("True", fontsize=10)
ax.set_title(f"Confusion matrix (val, row-normalized) — bal-acc {bal_acc:.3f}", fontsize=11)
fig.colorbar(im)
fig.tight_layout()
fig.savefig(OUT_DIR / "confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved confusion matrix to {OUT_DIR / 'confusion_matrix.png'}")

In [ ]:
# Plot training curves. x-axis is a running epoch index across all phases
# (head / ft / crt), with a dashed divider each time the phase changes.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
steps = list(range(len(history)))
boundaries = [i for i in range(1, len(history)) if history[i]["phase"] != history[i - 1]["phase"]]

# Loss curve
ax1.plot(steps, [h["loss"] for h in history], marker="o", linewidth=2)
for b in boundaries:
    ax1.axvline(b - 0.5, color="red", linestyle="--", alpha=0.5)
ax1.set_xlabel("Epoch (all phases)")
ax1.set_ylabel("Training Loss")
ax1.set_title("Training Loss")
ax1.grid(True, alpha=0.3)

# Balanced accuracy curve
ax2.plot(steps, [h["val_bal_acc"] for h in history], marker="o", linewidth=2, color="green")
for b in boundaries:
    ax2.axvline(b - 0.5, color="red", linestyle="--", alpha=0.5)
ax2.set_xlabel("Epoch (all phases)")
ax2.set_ylabel("Validation Balanced Accuracy")
ax2.set_title("Validation Balanced Accuracy")
ax2.grid(True, alpha=0.3)

fig.tight_layout()
fig.savefig(OUT_DIR / "training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved training curves to {OUT_DIR / 'training_curves.png'}")

## 9. Save Results

In [ ]:
# Save training log
with open(OUT_DIR / "training_log.json", "w") as f:
    json.dump({
        "model": MODEL_NAME,
        "img_size": IMG_SIZE,
        "classes": list(classes),
        "best_val_bal_acc": float(best),
        "history": history
    }, f, indent=2)

# Save class names
with open(OUT_DIR / "classes.json", "w") as f:
    json.dump(list(classes), f)

print(f"✓ Model checkpoint: {CKPT_PATH}")
print(f"✓ Model config: {OUT_DIR / 'model_config.json'}")
print(f"✓ Training log: {OUT_DIR / 'training_log.json'}")
print(f"✓ Confusion matrix: {OUT_DIR / 'confusion_matrix.png'}")
print(f"✓ Training curves: {OUT_DIR / 'training_curves.png'}")
print(f"\nAll results saved to {OUT_DIR}")
print("\nTo use the model locally (skin/03_app.py), download the contents of")
print("'galaxy-uq/results/skin/' from Google Drive into the repo's")
print("results/skin/ directory — the app reads model_config.json and adapts.")

## 10. Free memory before training another architecture

Run this cell **only when you want to try a different `MODEL_NAME` in the same
session**. It releases the model, optimizer, and data loaders and clears the
accelerator cache so a second backbone doesn't stack on top of the first and run
you out of memory — letting you iterate without a (slow) runtime restart. After
running it, change `MODEL_NAME` in the config cell and re-run from **section 4
(Data Loaders)** onward.

In [ ]:
import gc

# Release large objects so a second architecture doesn't stack on the first.
for _name in ["model", "opt", "criterion", "crt_criterion", "train_loader", "crt_loader",
              "val_loader", "preds", "targets"]:
    if _name in globals():
        del globals()[_name]

gc.collect()
if USE_TPU:
    # XLA has no explicit empty_cache; freeing refs + a step lets it reclaim device memory
    import torch_xla.core.xla_model as xm
    xm.mark_step()
elif torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Freed model/loaders and cleared the accelerator cache.")
print("Now change MODEL_NAME in the config cell and re-run from section 4 (Data Loaders).")